# 05 – Silver ilk full load

Bronze'daki `client`, `account`, `card`, `loan` tabloları tip dönüşümüyle silver'a yazılır. Her satıra `_change_ts` = 1998-12-31 00:00:00 eklenir. Silver `card`'da Change Data Feed açılır.

**Bu hücrenin altındaki kod:** `src` klasörünü `sys.path`'e ekler ve `silver_transforms` modülünü import eder.

**Çıktı:** `src path` repo'nun `src` klasörünü, `tables` 4 CDC tablosunu göstermeli.

In [0]:
import importlib
import os
import sys

repo_root = os.path.dirname(os.getcwd())
src_path = os.path.join(repo_root, "src")
if src_path not in sys.path:
    sys.path.append(src_path)

import silver_transforms
importlib.reload(silver_transforms)

print("src path:", src_path)
print("tables:", silver_transforms.CDC_TABLES)
print("initial _change_ts:", silver_transforms.INITIAL_CHANGE_TS)

## Dönüşüm testi (yazma yok)

4 bronze tablo okunur, `to_silver` ile silver tiplerine çevrilir ve `silver_dfs` dictionary'sinde tutulur. Silver'a hiçbir şey yazılmaz.

**Çıktı:** Her tablo için schema ve 2 örnek satır.

- ID kolonları `integer`, tarih kolonları `date` (`open_date`, `issued_date`, `loan_date`), `payments` `decimal(10,2)` olmalı.

- Tarihler 1993–1998 arasında olmalı (2093 gibi bir yıl çıkarsa tarih dönüşümü hatalı).

In [0]:
silver_dfs = {}

for table_name in silver_transforms.CDC_TABLES:
    bronze_table = silver_transforms.BRONZE_SCHEMA + "." + table_name
    df_bronze = spark.table(bronze_table)
    df_silver = silver_transforms.to_silver(df_bronze, table_name)
    silver_dfs[table_name] = df_silver

    print("=== " + table_name + " ===")
    df_silver.printSchema()
    df_silver.show(2, truncate=False)

## Silver ilk full load

Her tabloya `_change_ts` = 1998-12-31 00:00:00 eklenir ve `workspace.silver` altına Delta tablo olarak yazılır.

- `overwrite`: bu hücre tekrar çalışırsa silver bronze snapshot'ına döner, MERGE sonuçları silinir.

- `_change_ts`: MERGE'te gelen `change_ts` bu değerden büyükse update yapılır.

**Çıktı:** 4 silver tablonun adı ve satır sayısı. Beklenen: client 5369, account 4500, card 892, loan 682.

In [0]:
from pyspark.sql import functions as F

initial_change_ts = F.to_timestamp(F.lit(silver_transforms.INITIAL_CHANGE_TS))

for table_name in silver_transforms.CDC_TABLES:
    silver_table = silver_transforms.SILVER_SCHEMA + "." + table_name
    df_load = silver_dfs[table_name].withColumn("_change_ts", initial_change_ts)

    (
        df_load.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(silver_table)
    )

    row_count = spark.table(silver_table).count()
    print(silver_table, row_count)

## Silver `card`'da Change Data Feed

Kart iptali silver'da hard delete olduğu için iptal edilen kartlar CDF ile izlenir (08_silver_history'de `table_changes()` ile sorgulanır).

- Yukarıdaki full load her çalıştığında tablo yeniden oluşur, bu hücre de ondan sonra tekrar çalıştırılır.

**Çıktı:** `delta.enableChangeDataFeed` = `true` satırı görünmeli.

In [0]:
%sql
ALTER TABLE workspace.silver.card SET TBLPROPERTIES (delta.enableChangeDataFeed = true);

SHOW TBLPROPERTIES workspace.silver.card;

## Kontrol 1: NULL sayıları (cast hatası)

`cast` ve `to_date` dönüştüremediği değeri hata vermeden NULL yapar. Bu hücre 4 silver tablonun her kolonundaki NULL sayısını gösterir.

**Çıktı:** Her tablo için tek satır; her kolon bir NULL sayısı. Hepsi 0 olmalı.

In [0]:
for table_name in silver_transforms.CDC_TABLES:
    silver_table = silver_transforms.SILVER_SCHEMA + "." + table_name
    df_check = spark.table(silver_table)

    null_counts = []
    for column_name in df_check.columns:
        null_count = F.count_if(F.col(column_name).isNull()).alias(column_name)
        null_counts.append(null_count)

    print("=== " + table_name + " ===")
    df_check.select(null_counts).show(truncate=False)


## Kontrol 2: Tarih aralıkları

`"19"` + `YYMMDD` dönüşümünün doğru çalıştığını kontrol eder. Berka 1993–1998 arasını kapsar.

**Çıktı:** Her tarih kolonu için `min_date` ve `max_date`. Hepsi 1993-01-01 ile 1998-12-31 arasında olmalı (20xx yılı çıkmamalı).

In [0]:
date_columns = {
    "account": "open_date",
    "card": "issued_date",
    "loan": "loan_date",
}

for table_name in date_columns:
    column_name = date_columns[table_name]
    silver_table = silver_transforms.SILVER_SCHEMA + "." + table_name
    df_check = spark.table(silver_table)

    df_range = df_check.select(
        F.lit(table_name).alias("table"),
        F.lit(column_name).alias("column"),
        F.min(column_name).alias("min_date"),
        F.max(column_name).alias("max_date"),
    )
    df_range.show(truncate=False)